# PTB-XL ECG Diagnosis — Ablation Study (Colab)

Reproduces part of the benchmark from:
> Nonaka, K., & Seita, D. (2021). *In-depth Benchmarking of Deep Neural Network Architectures for ECG Diagnosis.*

**Ablation dimensions:** Task type (multilabel vs multiclass) × Model (MLP, CNN, Transformer) × Hidden dim (32, 64, 128)

**Authors:** Ankita Jain (ankitaj3@illinois.edu), Manish Singh (manishs4@illinois.edu)

## 0. Install & Restart
Run this cell. Runtime will auto-restart. Then click Runtime → Run all again.

In [ ]:
!pip uninstall -y numpy
!pip install -q "numpy==2.2.0" pyhealth wfdb requests
import os; os.kill(os.getpid(), 9)

Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.2/91.2 kB 5.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.1/16.1 MB 95.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 622.3/622.3 kB 30.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 163.9/163.9 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 62.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 205.4/205.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.4/7.4 MB 82.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

## 1. Mount Drive & Inject Custom PTB-XL Modules

In [1]:
from google.colab import drive
drive.mount('/content/drive')

ROOT = '/content/drive/MyDrive/pyhealth'

import os
assert os.path.exists(os.path.join(ROOT, 'ptbxl_database.csv')), f'Data not found at {ROOT}'
print(f'PTB-XL data found at: {ROOT}')

Mounted at /content/drive
PTB-XL data found at: /content/drive/MyDrive/pyhealth


In [2]:
# Inject custom PTBXLDataset into pip-installed pyhealth
import pyhealth, os, ast, logging
import pandas as pd
from pathlib import Path
from typing import List, Optional, Any, Dict

pkg = os.path.dirname(pyhealth.__file__)

# --- Write ptbxl.yaml config ---
cfg_dir = os.path.join(pkg, 'datasets', 'configs')
os.makedirs(cfg_dir, exist_ok=True)
with open(os.path.join(cfg_dir, 'ptbxl.yaml'), 'w') as f:
    f.write('''version: "1.0.0"
tables:
  ptbxl:
    file_path: "ptbxl-metadata-pyhealth.csv"
    patient_id: "patient_id"
    timestamp: null
    attributes:
      - "record_id"
      - "signal_file"
      - "scp_codes"
      - "sampling_rate"
      - "num_leads"
''')
print('Config written.')

Config written.


In [3]:
# --- Write ptbxl.py dataset module ---
ptbxl_dataset_code = '''
import logging, os
from pathlib import Path
from typing import List, Optional
import pandas as pd
from .base_dataset import BaseDataset

logger = logging.getLogger(__name__)

class PTBXLDataset(BaseDataset):
    SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

    def __init__(self, root, sampling_rate=100, dataset_name=None,
                 config_path=None, cache_dir=None, num_workers=1, dev=False):
        if sampling_rate not in (100, 500):
            raise ValueError(f"sampling_rate must be 100 or 500, got {sampling_rate}.")
        self.sampling_rate = sampling_rate
        if config_path is None:
            config_path = Path(__file__).parent / "configs" / "ptbxl.yaml"
        metadata_csv = os.path.join(root, "ptbxl-metadata-pyhealth.csv")
        if not os.path.exists(metadata_csv):
            self.prepare_metadata(root)
        super().__init__(root=root, tables=["ptbxl"], dataset_name=dataset_name or "ptbxl",
                         config_path=config_path, cache_dir=cache_dir,
                         num_workers=num_workers, dev=dev)

    def prepare_metadata(self, root=None):
        root = root or self.root
        db_path = os.path.join(root, "ptbxl_database.csv")
        if not os.path.exists(db_path):
            raise FileNotFoundError(f"ptbxl_database.csv not found in {root}")
        df = pd.read_csv(db_path, index_col="ecg_id")
        rate_col = "filename_lr" if self.sampling_rate == 100 else "filename_hr"
        records = []
        for ecg_id, row in df.iterrows():
            records.append({
                "patient_id": str(int(row["patient_id"])),
                "record_id": int(ecg_id),
                "signal_file": str(row[rate_col]),
                "scp_codes": str(row.get("scp_codes", "{}")),
                "sampling_rate": self.sampling_rate,
                "num_leads": 12,
            })
        out_df = pd.DataFrame(records)
        out_df.to_csv(os.path.join(root, "ptbxl-metadata-pyhealth.csv"), index=False)
        logger.info(f"Wrote metadata ({len(out_df)} records).")
'''

with open(os.path.join(pkg, 'datasets', 'ptbxl.py'), 'w') as f:
    f.write(ptbxl_dataset_code)
print('PTBXLDataset module written.')

PTBXLDataset module written.


In [4]:
# --- Write ptbxl_diagnosis.py task module ---
ptbxl_task_code = '''
import ast, logging
from typing import Any, Dict, List
from .base_task import BaseTask

logger = logging.getLogger(__name__)
SUPERCLASSES = ["NORM", "MI", "STTC", "CD", "HYP"]

SCP_TO_SUPER = {
    "NORM": "NORM", "IMI": "MI", "ASMI": "MI", "ILMI": "MI", "AMI": "MI",
    "ALMI": "MI", "INJAS": "MI", "LMI": "MI", "INJAL": "MI", "IPLMI": "MI",
    "IPMI": "MI", "INJIN": "MI", "INJLA": "MI", "RMI": "MI", "INJIL": "MI",
    "STD_": "STTC", "ISCA": "STTC", "ISCI": "STTC", "ISC_": "STTC",
    "IVCTE": "STTC", "STTC": "STTC", "NST_": "STTC", "STE_": "STTC",
    "LNGQT": "STTC", "TAB_": "STTC", "INVT": "STTC",
    "LVOLT": "HYP", "HVOLT": "HYP", "HYP": "HYP", "RVH": "HYP",
    "LVH": "HYP", "LAO/LAE": "HYP", "RAO/RAE": "HYP", "SEHYP": "HYP",
    "LAFB/LPFB": "CD", "IRBBB": "CD", "ILBBB": "CD", "CRBBB": "CD",
    "CLBBB": "CD", "IVCD": "CD", "LBBB": "CD", "RBBB": "CD",
    "WPW": "CD", "LPFB": "CD", "LAFB": "CD", "CD": "CD",
}

def _scp_to_superclasses(scp_codes_str):
    try:
        codes = ast.literal_eval(scp_codes_str)
    except (ValueError, SyntaxError):
        return []
    supers = set()
    for code, likelihood in codes.items():
        if likelihood > 0 and code in SCP_TO_SUPER:
            supers.add(SCP_TO_SUPER[code])
    return sorted(supers)

class PTBXLDiagnosis(BaseTask):
    task_name = "PTBXLDiagnosis"
    input_schema = {"signal_file": "signal_file"}
    output_schema = {"labels": "multilabel"}
    def __call__(self, patient):
        events = patient.get_events(event_type="ptbxl")
        samples = []
        for event in events:
            scp_raw = event["scp_codes"] if "scp_codes" in event else "{}"
            labels = _scp_to_superclasses(str(scp_raw))
            if not labels: continue
            samples.append({"patient_id": patient.patient_id,
                "record_id": event["record_id"] if "record_id" in event else None,
                "signal_file": str(event["signal_file"]) if "signal_file" in event else "",
                "labels": labels})
        return samples

class PTBXLMulticlassDiagnosis(BaseTask):
    task_name = "PTBXLMulticlassDiagnosis"
    input_schema = {"signal_file": "signal_file"}
    output_schema = {"label": "multiclass"}
    def __call__(self, patient):
        events = patient.get_events(event_type="ptbxl")
        samples = []
        for event in events:
            scp_str = str(event["scp_codes"]) if "scp_codes" in event else "{}"
            try: codes = ast.literal_eval(scp_str)
            except (ValueError, SyntaxError): continue
            scores = {}
            for code, likelihood in codes.items():
                if likelihood > 0 and code in SCP_TO_SUPER:
                    sup = SCP_TO_SUPER[code]
                    scores[sup] = scores.get(sup, 0.0) + likelihood
            if not scores: continue
            dominant = max(scores, key=lambda k: scores[k])
            samples.append({"patient_id": patient.patient_id,
                "record_id": event["record_id"] if "record_id" in event else None,
                "signal_file": str(event["signal_file"]) if "signal_file" in event else "",
                "label": dominant})
        return samples
'''

with open(os.path.join(pkg, 'tasks', 'ptbxl_diagnosis.py'), 'w') as f:
    f.write(ptbxl_task_code)
print('Task modules written.')
print('Custom modules injected into pyhealth!')

Task modules written.
Custom modules injected into pyhealth!


## 2. Imports & Setup

In [5]:
import numpy as np
import torch
import wfdb
import os
from collections import Counter

torch.manual_seed(42)
np.random.seed(42)

from pyhealth.datasets.ptbxl import PTBXLDataset
from pyhealth.tasks.ptbxl_diagnosis import (
    PTBXLDiagnosis, PTBXLMulticlassDiagnosis, SUPERCLASSES,
)
from pyhealth.datasets import create_sample_dataset, get_dataloader, split_by_sample
from pyhealth.models import MLP, CNN, Transformer
from pyhealth.trainer import Trainer

print('All imports successful!')

All imports successful!


## 3. Load Dataset & Generate Task Samples

In [6]:
dataset = PTBXLDataset(root=ROOT, sampling_rate=500)

ml_samples, mc_samples = [], []
ml_task, mc_task = PTBXLDiagnosis(), PTBXLMulticlassDiagnosis()

for patient in dataset.iter_patients():
    ml_samples.extend(ml_task(patient))
    mc_samples.extend(mc_task(patient))

print(f'Multilabel samples : {len(ml_samples)}')
print(f'Multiclass samples : {len(mc_samples)}')

Initializing ptbxl dataset from /content/drive/MyDrive/pyhealth (dev mode: False)


INFO:pyhealth.datasets.base_dataset:Initializing ptbxl dataset from /content/drive/MyDrive/pyhealth (dev mode: False)


No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/4479aeea-da26-50cb-945b-dfd94f2caf84


INFO:pyhealth.datasets.base_dataset:No cache_dir provided. Using default cache dir: /root/.cache/pyhealth/4479aeea-da26-50cb-945b-dfd94f2caf84


No cached event dataframe found. Creating: /root/.cache/pyhealth/4479aeea-da26-50cb-945b-dfd94f2caf84/global_event_df.parquet


...

Scanning table: ptbxl from /content/drive/MyDrive/pyhealth/ptbxl-metadata-pyhealth.csv


INFO:pyhealth.datasets.base_dataset:Scanning table: ptbxl from /content/drive/MyDrive/pyhealth/ptbxl-metadata-pyhealth.csv


Caching event dataframe to /root/.cache/pyhealth/4479aeea-da26-50cb-945b-dfd94f2caf84/global_event_df.parquet...


INFO:pyhealth.datasets.base_dataset:Caching event dataframe to /root/.cache/pyhealth/4479aeea-da26-50cb-945b-dfd94f2caf84/global_event_df.parquet...


Multilabel samples : 19432
Multiclass samples : 19432


## 4. Label Distribution

In [7]:
ml_dist = Counter(l for s in ml_samples for l in s['labels'])
mc_dist = Counter(s['label'] for s in mc_samples)
print('Multilabel distribution:', dict(sorted(ml_dist.items())))
print('Multiclass distribution:', dict(sorted(mc_dist.items())))

Multilabel distribution: {'CD': 4469, 'HYP': 2649, 'MI': 5458, 'NORM': 9514, 'STTC': 1936}
Multiclass distribution: {'CD': 3316, 'HYP': 1495, 'MI': 4404, 'NORM': 9271, 'STTC': 946}


## 5. Load ECG Signals & Build SampleDataset

In [8]:
def load_ecg_signal(root, signal_file):
    try:
        signal, _ = wfdb.rdsamp(os.path.join(root, signal_file))
        return signal.flatten().tolist()
    except Exception:
        return None

def build_dataset(task_samples, root, task_type='multiclass'):
    feature_samples, skipped = [], 0
    total = len(task_samples)
    for i, s in enumerate(task_samples):
        if (i+1) % 2000 == 0:
            print(f'\r  Loading: {i+1}/{total}', end='', flush=True)
        signal = load_ecg_signal(root, s['signal_file'])
        if signal is None: skipped += 1; continue
        sample = {'patient_id': s['patient_id'], 'visit_id': str(s['record_id']), 'ecg_signal': signal}
        if task_type == 'multilabel': sample['labels'] = s['labels']
        else: sample['label'] = s['label']
        feature_samples.append(sample)
    print(f'\r  Loaded {len(feature_samples)} signals ({skipped} skipped)')
    label_key = 'labels' if task_type == 'multilabel' else 'label'
    label_mode = 'multilabel' if task_type == 'multilabel' else 'multiclass'
    return create_sample_dataset(samples=feature_samples,
        input_schema={'ecg_signal': 'sequence'},
        output_schema={label_key: label_mode}, dataset_name='ptbxl_real')

In [9]:
import random
random.seed(42)

print('Building multiclass dataset (500 samples) ...')
mc_ds = build_dataset(random.sample(mc_samples, 500), ROOT, task_type='multiclass')

print('Building multilabel dataset (500 samples) ...')
ml_ds = build_dataset(random.sample(ml_samples, 500), ROOT, task_type='multilabel')


Building multiclass dataset (500 samples) ...
  Loaded 500 signals (0 skipped)
Label label vocab: {'CD': 0, 'HYP': 1, 'MI': 2, 'NORM': 3, 'STTC': 4}


INFO:pyhealth.processors.label_processor:Label label vocab: {'CD': 0, 'HYP': 1, 'MI': 2, 'NORM': 3, 'STTC': 4}


Building multilabel dataset (500 samples) ...
  Loaded 500 signals (0 skipped)
Label labels vocab: {'CD': 0, 'HYP': 1, 'MI': 2, 'NORM': 3, 'STTC': 4}


INFO:pyhealth.processors.label_processor:Label labels vocab: {'CD': 0, 'HYP': 1, 'MI': 2, 'NORM': 3, 'STTC': 4}


## 6. Ablation — Model × Hidden Dim

In [10]:
EPOCHS, BATCH_SIZE, LR = 10, 32, 1e-3
HIDDEN_DIMS = [32, 64, 128]
MODEL_CONFIGS = [(MLP, 'MLP'), (CNN, 'CNN'), (Transformer, 'Transformer')]

def run_ablation(sample_ds, model_cls, hidden_dim, metrics_list):
    train_ds, val_ds, test_ds = split_by_sample(sample_ds, [0.7, 0.15, 0.15])
    train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = get_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
    test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False)
    if model_cls is MLP:
        model = model_cls(dataset=sample_ds, hidden_dim=hidden_dim, n_layers=2)
    elif model_cls is CNN:
        model = model_cls(dataset=sample_ds, hidden_dim=hidden_dim, num_layers=2)
    elif model_cls is Transformer:
        model = model_cls(dataset=sample_ds, embedding_dim=hidden_dim, num_layers=2)
    trainer = Trainer(model=model, metrics=metrics_list, enable_logging=False)
    trainer.train(train_dataloader=train_loader, val_dataloader=val_loader,
                  epochs=EPOCHS, optimizer_params={'lr': LR})
    return {'val': trainer.evaluate(val_loader), 'test': trainer.evaluate(test_loader)}

### 6a. Multiclass (F1, Accuracy)

In [11]:
mc_metrics = ['f1_weighted', 'accuracy']
mc_results = []
print(f"{'Model':<14} {'hidden_dim':<12} {'val_f1':<10} {'test_f1':<10} {'test_acc':<10} {'test_loss'}")
print('-' * 68)
for model_cls, name in MODEL_CONFIGS:
    for hd in HIDDEN_DIMS:
        try:
            r = run_ablation(mc_ds, model_cls, hd, mc_metrics)
            v, t = r['val'], r['test']
            print(f"{name:<14} {hd:<12} {v.get('f1_weighted',0):<10.4f} {t.get('f1_weighted',0):<10.4f} {t.get('accuracy',0):<10.4f} {t.get('loss',0):.4f}")
            mc_results.append({'model': name, 'hidden_dim': hd, **t})
        except Exception as e:
            print(f"{name:<14} {hd:<12} FAILED: {e}")

Model          hidden_dim   val_f1     test_f1    test_acc   test_loss
--------------------------------------------------------------------
MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (ecg_signal): Embedding(6843, 128, padding_idx=0)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (ecg_signal): Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (fc): Linear(in_features=32, out_features=5, bias=True)
)


INFO:pyhealth.trainer:MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (ecg_signal): Embedding(6843, 128, padding_idx=0)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (ecg_signal): Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (fc): Linear(in_features=32, out_features=5, bias=True)
)


Metrics: ['f1_weighted', 'accuracy']


INFO:pyhealth.trainer:Metrics: ['f1_weighted', 'accuracy']


Device: cpu


...

accuracy: 0.4133


INFO:pyhealth.trainer:accuracy: 0.4133


loss: 1.6364


INFO:pyhealth.trainer:loss: 1.6364
Evaluation: 100%|██████████| 3/3 [00:04<00:00,  1.53s/it]

Transformer    128          0.3846     0.4284     0.4267     2.0862


### 6b. Multilabel (ROC-AUC, F1)

In [12]:
ml_metrics = ['roc_auc_samples', 'f1_weighted']
ml_results = []
print(f"{'Model':<14} {'hidden_dim':<12} {'val_auroc':<12} {'test_auroc':<12} {'test_f1':<10} {'test_loss'}")
print('-' * 72)
for model_cls, name in MODEL_CONFIGS:
    for hd in HIDDEN_DIMS:
        try:
            r = run_ablation(ml_ds, model_cls, hd, ml_metrics)
            v, t = r['val'], r['test']
            print(f"{name:<14} {hd:<12} {v.get('roc_auc_samples',0):<12.4f} {t.get('roc_auc_samples',0):<12.4f} {t.get('f1_weighted',0):<10.4f} {t.get('loss',0):.4f}")
            ml_results.append({'model': name, 'hidden_dim': hd, **t})
        except Exception as e:
            print(f"{name:<14} {hd:<12} FAILED: {e}")

Model          hidden_dim   val_auroc    test_auroc   test_f1    test_loss
------------------------------------------------------------------------
MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (ecg_signal): Embedding(6964, 128, padding_idx=0)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (ecg_signal): Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (fc): Linear(in_features=32, out_features=5, bias=True)
)


INFO:pyhealth.trainer:MLP(
  (embedding_model): EmbeddingModel(embedding_layers=ModuleDict(
    (ecg_signal): Embedding(6964, 128, padding_idx=0)
  ))
  (activation): ReLU()
  (mlp): ModuleDict(
    (ecg_signal): Sequential(
      (0): Linear(in_features=128, out_features=32, bias=True)
      (1): ReLU()
      (2): Linear(in_features=32, out_features=32, bias=True)
    )
  )
  (fc): Linear(in_features=32, out_features=5, bias=True)
)


Metrics: ['roc_auc_samples', 'f1_weighted']


INFO:pyhealth.trainer:Metrics: ['roc_auc_samples', 'f1_weighted']


Device: cpu


...

f1_weighted: 0.4111


INFO:pyhealth.trainer:f1_weighted: 0.4111


loss: 0.7565


INFO:pyhealth.trainer:loss: 0.7565
Evaluation: 100%|██████████| 3/3 [00:04<00:00,  1.47s/it]

Transformer    128          0.6989       0.6378       0.4001     0.6752


## 7. SCP Code Coverage

In [13]:
from pyhealth.tasks.ptbxl_diagnosis import SCP_TO_SUPER
print(f'Total mapped SCP codes : {len(SCP_TO_SUPER)}')
print(f'Superclasses covered   : {sorted(set(SCP_TO_SUPER.values()))}')

Total mapped SCP codes : 46
Superclasses covered   : ['CD', 'HYP', 'MI', 'NORM', 'STTC']


## 8. Summary

**Reference:** Nonaka & Seita (2021): Multilabel ROC-AUC ~0.93 (ResNet), Multiclass F1 ~0.82 (ResNet)

## 9. Ablation Ideas Considered but Not Used
1. **Bandpass Filtering** — PTB-XL already preprocessed
2. **Lead Subset (12 vs 6 vs 1)** — Outside paper scope
3. **Patient-level splitting** — Deprioritized for model architecture ablations

In [14]:
mc_ds.close()
ml_ds.close()
print('Done.')

Done.
